# Implementación ETL - RaSA

Este cuaderno implementa el diseño ETL acordado para RaSA. Ejecutar las celdas en orden: primero configurar Spark, después extraer y transformar las fuentes, y finalmente cargar dimensiones y hechos.

La carga es segura para una primera ejecución: si una tabla destino ya contiene registros, no inserta filas adicionales.

## Alcance

Se cargan las dimensiones de áreas de servicio, geografía, nivel de servicio, condición de pago, fecha, tipo de beneficio y proveedor; la minidimensión de condiciones; la asociación área-geografía; la tabla factless de historia y el hecho de planes-tipos de beneficio.

Todas las dimensiones incorporan un registro desconocido con llave DWH igual a 0. Así se preservan hechos cuya llave de área de servicio u otra dimensión no tenga correspondencia válida.

## 1. Configuración de la conexión

In [1]:
# Configuración servidor base de datos transaccional
# Recuerde usar Estudiante_i como usuario y la contraseña asigana en el excel de conexión a maquina virtual como contraseña
db_user = 'DB_202613_j_rodriguezv23'
db_psswd = '201728876'
source_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/RaSaTransaccional_ETL'

dest_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/DB_202613_j_rodriguezv23'

# Driver de conexion
import tarfile
import urllib.request
import os 

def descargar_asset(url: str, ruta_destino: str = './content/assets') -> str:
    os.makedirs(ruta_destino, exist_ok=True)

    nombre_archivo = url.split('/')[-1]
    ruta_completa_archivo = os.path.join(ruta_destino, nombre_archivo)

    urllib.request.urlretrieve(url, ruta_completa_archivo)

    if nombre_archivo.endswith('.tar.gz'):
        try:
            with tarfile.open(ruta_completa_archivo, "r:gz") as tar:
                jar_member = None
                for member in tar.getmembers():
                    if member.name.endswith('.jar'):
                        jar_member = member
                        break
                if jar_member:
                    nombre_jar = os.path.basename(jar_member.name)
                    ruta_jar_extraido = os.path.join(ruta_destino, nombre_jar)
                    with tar.extractfile(jar_member) as f_in:
                        with open(ruta_jar_extraido, 'wb') as f_out:
                            f_out.write(f_in.read())
                    return ruta_jar_extraido
                else:
                    return ruta_completa_archivo
        except tarfile.ReadError:
            return ruta_completa_archivo
    else:
        return ruta_completa_archivo

url = 'https://dev.mysql.com/get/Downloads/Connector-J/mysql-connector-j-9.3.0.tar.gz'

path_jar_driver = descargar_asset(url)

# path_jar_driver = 'C:\Program Files (x86)\MySQL\Connector J 8.0\mysql-connector-java-8.0.28.jar'
SOURCE_SCHEMA = "RaSaTransaccional_ETL"
DEST_SCHEMA = dest_db_connection_string.rsplit("/", 1)[-1]

In [2]:
from pyspark.sql import SparkSession, Window, functions as f
from datetime import date
import re

spark = (
    SparkSession.builder
    .appName("ETL_RaSA")
    .config("spark.driver.extraClassPath", path_jar_driver)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

SOURCE_TABLES = {
    "areas": ["FuenteAreasDeServicio_ETL"],
    "beneficios": ["FuenteTiposBeneficio_ETL"],
    "planes": ["FuentePlanesBeneficio_ETL"],
    "niveles": ["NivelesDeServicio"],
    "pagos": ["FuenteCondicionesDePago_ETL"],
}

DEST = {
    "mini": "DimMiniCondicionesTipoBeneficio",
    "areas": "DimAreasDeServicio",
    "geografia": "DimGeografia",
    "asociacion": "DimAsociacionAreaServicioGeografia",
    "niveles": "DimNivelesDeServicio",
    "pagos": "DimCondicionDePago",
    "fechas": "DimFecha",
    "tipos": "DimTipoBeneficio",
    "proveedores": "DimProveedor",
    "historia": "HechoHistCondicionesTiposBeneficio",
    "hecho": "HechoPlanesTipoBeneficio",
}

Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
26/07/15 17:47:14 WARN Utils: Your hostname, codespaces-f4ccda resolves to a loopback address: 127.0.0.1; using 10.0.14.114 instead (on interface eth0)
26/07/15 17:47:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/15 17:47:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Utilidades

In [3]:
def leer_bd(connection, sql):
    return (
        spark.read.format("jdbc")
        .option("url", connection)
        .option("dbtable", sql)
        .option("user", db_user)
        .option("password", db_psswd)
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .load()
    )


def resolver_tabla(candidatas):
    lista = ", ".join("'" + x.lower() + "'" for x in candidatas)
    sql = f"(SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '{SOURCE_SCHEMA}' AND LOWER(TABLE_NAME) IN ({lista})) AS catalogo"
    encontradas = [x["TABLE_NAME"] for x in leer_bd(source_db_connection_string, sql).collect()]
    if not encontradas:
        raise ValueError(f"No se encontró ninguna tabla de {candidatas} en {SOURCE_SCHEMA}.")
    return encontradas[0]


def leer_fuente(nombre):
    tabla = resolver_tabla(SOURCE_TABLES[nombre])
    print(f"Extrayendo {SOURCE_SCHEMA}.{tabla}")
    return leer_bd(source_db_connection_string, f"(SELECT * FROM {SOURCE_SCHEMA}.{tabla}) AS {nombre}")


def canon(nombre):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(nombre).lower().translate(str.maketrans("áéíóúüñ", "aeiouun")),
    )


def c(df, *alternativas):
    disponibles = {canon(x): x for x in df.columns}
    for alternativa in alternativas:
        if canon(alternativa) in disponibles:
            return f.col(disponibles[canon(alternativa)])
    raise ValueError(f"No se halló {alternativas}. Disponibles: {df.columns}")


def fecha(columna):
    texto = f.trim(columna.cast("string"))
    return f.coalesce(
        f.to_date(texto, "yyyy-MM-dd"),
        f.to_date(texto, "yyyy/MM/dd"),
        f.to_date(texto, "MM/dd/yyyy"),
        f.to_date(texto, "MMM dd,yyyy"),
        f.to_date(f.when(texto.rlike(r"^[0-9]{4}-[0-9]{1,2}$"), f.concat(texto, f.lit("-01"))), "yyyy-MM-dd"),
        f.to_date(f.when(texto.rlike(r"^[0-9]{4}$"), f.concat(texto, f.lit("-01-01"))), "yyyy-MM-dd"),
    )


def si_no(columna):
    valor = f.upper(f.trim(columna.cast("string")))
    return (
        f.when(valor.isin("YES", "Y", "SI", "SÍ", "1", "TRUE"), "Yes")
        .when(valor.isin("NO", "N", "0", "FALSE"), "No")
        .otherwise(None)
    )


def tipo_pago(columna):
    valor = f.upper(f.trim(columna.cast("string")))
    return (
        f.when(valor.contains("COSEGURO"), "Coseguro")
        .when(valor.contains("COPAGO"), "Copago")
        .otherwise(None)
    )


def desconocido(df, valores):
    return spark.createDataFrame([valores], df.schema).unionByName(df)


def destino_tiene_datos(tabla):
    sql = f"(SELECT COUNT(*) AS n FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '{DEST_SCHEMA}' AND TABLE_NAME = '{tabla}') AS existe"
    if leer_bd(dest_db_connection_string, sql).first()["n"] == 0:
        return False
    return leer_bd(
        dest_db_connection_string,
        f"(SELECT 1 AS x FROM {DEST_SCHEMA}.{tabla} LIMIT 1) AS muestra",
    ).count() > 0

'''
def cargar_si_vacia(df, tabla, batchsize=1000):
    if destino_tiene_datos(tabla):
        print(f"{tabla}: se omite porque ya tiene datos.")
        return
    (
        df.write.format("jdbc")
        .mode("append")
        .option("url", dest_db_connection_string)
        .option("dbtable", f"{DEST_SCHEMA}.{tabla}")
        .option("user", db_user)
        .option("password", db_psswd)
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .option("batchsize", batchsize)
        .save()
    )
    print(f"{tabla}: {df.count()} filas cargadas.")
    
'''

spark.conf.set("spark.sql.shuffle.partitions", "8")

def url_lotes(url):
    separador = "&" if "?" in url else "?"
    return f"{url}{separador}rewriteBatchedStatements=true&useServerPrepStmts=false"

DEST_JDBC_LOTES = url_lotes(dest_db_connection_string)

def cargar_si_vacia(df, tabla, batchsize=10_000, particiones=1):
    if destino_tiene_datos(tabla):
        print(f"{tabla}: se omite porque ya tiene datos.")
        return

    df_lote = df.coalesce(particiones)

    (
        df_lote.write.format("jdbc")
        .mode("append")
        .option("url", DEST_JDBC_LOTES)
        .option("dbtable", f"{DEST_SCHEMA}.{tabla}")
        .option("user", db_user)
        .option("password", db_psswd)
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .option("batchsize", batchsize)
        .option("numPartitions", particiones)
        .save()
    )

    print(f"{tabla}: carga completada.")

## 3. Extracción y limpieza de fuentes

In [4]:
areas_raw = leer_fuente("areas")
beneficios_raw = leer_fuente("beneficios")
planes_raw = leer_fuente("planes")
niveles_raw = leer_fuente("niveles")
pagos_raw = leer_fuente("pagos")

for nombre, df in {
    "Áreas": areas_raw,
    "Beneficios": beneficios_raw,
    "Planes-beneficios": planes_raw,
    "Niveles": niveles_raw,
    "Condiciones de pago": pagos_raw,
}.items():
    print(nombre, df.count(), df.columns)

Extrayendo RaSaTransaccional_ETL.FuenteAreasDeServicio_ETL
Extrayendo RaSaTransaccional_ETL.FuenteTiposBeneficio_ETL
Extrayendo RaSaTransaccional_ETL.FuentePlanesBeneficio_ETL
Extrayendo RaSaTransaccional_ETL.NivelesDeServicio


Extrayendo RaSaTransaccional_ETL.FuenteCondicionesDePago_ETL


Áreas 268444 ['IdAreaDeServicio_T', 'NombreAreaDeServicio', 'IdGeografia_T', 'Condado', 'Estado', 'PoblacionAct', 'Area', 'Densidad', 'Fecha']


Beneficios 449 ['IdTipoBeneficio_T', 'Nombre', 'UnidadDelLimite', 'EsEHB', 'EstaCubiertaPorSeguro', 'TieneLimiteCuantitativo', 'ExcluidoDelDesembolsoMaximoDentroDeLaRed', 'ExcluidoDelDesembolsoMaximoFueraDeLaRed', 'Fecha']


Planes-beneficios 70752 ['IdTipoBeneficio_T', 'IdAreaDeServicio_T', 'IdCondicionDePagoCopago_T', 'IdCondicionDePagoCoseguro_T', 'IdNivelServicio_T', 'IdPlan_T', 'Fecha', 'IdProveedor_T', 'valorCopago', 'valorCoseguro', 'cantidadLimite']


Niveles 3 ['IdNivelDeServicio_DWH', 'IdNivelDeServicio_T', 'Descripcion']
Condiciones de pago 26 ['IdCondicionesDePago_T', 'Descripcion', 'Tipo']


In [5]:
areas = (
    areas_raw.select(
        c(areas_raw, "idAreaDeServicio_T", "idAreaServicio_T", "idAreaDeServicio", "idAreaServicio").cast("long").alias("IdAreaDeServicio_T"),
        c(areas_raw, "NombreAreaDeServicio", "Nombre").cast("string").alias("NombreAreaDeServicio"),
        fecha(c(areas_raw, "Fecha")).alias("Fecha"),
        c(areas_raw, "idGeografia_T", "idGeografía_T", "idGeografia", "idGeografía").cast("long").alias("IdGeografia_T"),
        c(areas_raw, "Estado").cast("string").alias("Estado"),
        c(areas_raw, "Condado").cast("string").alias("Condado"),
        c(areas_raw, "PoblacionAct", "PoblacionActual").cast("string").alias("PoblacionTexto"),
        c(areas_raw, "Area").cast("double").alias("Area"),
        c(areas_raw, "Densidad").cast("double").alias("Densidad"),
    )
    .dropDuplicates()
    .withColumn("Area", f.abs("Area"))
    .withColumn("PoblacionAct", f.regexp_replace("PoblacionTexto", r"0001$", "").cast("long"))
    .drop("PoblacionTexto")
)

beneficios = (
    beneficios_raw.select(
        c(beneficios_raw, "idTipoBeneficio_T", "idTipoBeneficio").cast("long").alias("IdTipoBeneficio_T"),
        c(beneficios_raw, "Nombre").cast("string").alias("Nombre"),
        c(beneficios_raw, "UnidadDelLimite", "UnidadDeLimite").cast("string").alias("UnidadDelLimite"),
        si_no(c(beneficios_raw, "EsEHB")).alias("EsEHB"),
        si_no(c(beneficios_raw, "EstaCubiertoPorSeguro", "EstaCubiertaPorSeguro")).alias("EstaCubiertoPorSeguro"),
        si_no(c(beneficios_raw, "TieneLimiteCuantitativo")).alias("TieneLimiteCuantitativo"),
        si_no(c(beneficios_raw, "ExcluidoDelDesembolsoMaximoDentroDeLaRed")).alias("ExcluidoDelDesembolsoMaximoDentroDeLaRed"),
        si_no(c(beneficios_raw, "ExcluidoDelDesembolsoMaximoFueraDeLaRed")).alias("ExcluidoDelDesembolsoMaximoFueraDeLaRed"),
        fecha(c(beneficios_raw, "Fecha")).alias("FechaInicio"),
    )
    .filter(f.col("IdTipoBeneficio_T").isNotNull() & f.col("FechaInicio").isNotNull())
    .dropDuplicates()
)

dim_pago_fuente = (
    pagos_raw.select(
        c(pagos_raw, "idCondicionesDePago_T", "idCondicionDePago_T", "idCondicionesDePago", "idCondicionDePago").cast("long").alias("IdCondicionesDePago_T"),
        c(pagos_raw, "Descripcion", "Descripción").cast("string").alias("Descripcion"),
        tipo_pago(c(pagos_raw, "Tipo")).alias("Tipo"),
    )
    .filter(f.col("Tipo").isNotNull())
    .dropDuplicates()
)

niveles_fuente = (
    niveles_raw.select(
        c(niveles_raw, "idNivelDeServicio_T", "idNivelServicio_T", "idNivelDeRed_T", "idNivelDeRed", "idNivelServicio").cast("long").alias("IdNivelDeServicio_T"),
        c(niveles_raw, "Descripcion", "Descripción", "Nombre", "NivelDeRed").cast("string").alias("Descripcion"),
    )
    .dropDuplicates()
)

planes = (
    planes_raw.select(
        c(planes_raw, "idPlan_T", "idPlan").cast("long").alias("IdPlan"),
        c(planes_raw, "idTipoBeneficio_T", "idTipoBeneficio").cast("long").alias("IdTipoBeneficio_T"),
        c(planes_raw, "idAreaDeServicio_T", "idAreaServicio_T", "idAreaDeServicio", "idAreaServicio").cast("long").alias("IdAreaDeServicio_T"),
        c(planes_raw, "idCondicionesDePagoCopago_T", "idCondicionDePagoCopago_T", "idCondicionesDePagoCopago").cast("long").alias("IdCondicionesDePagoCopago_T"),
        c(planes_raw, "idCondicionesDePagoCoseguro_T", "idCondicionDePagoCoseguro_T", "idCondicionesDePagoCoseguro").cast("long").alias("IdCondicionesDePagoCoseguro_T"),
        c(planes_raw, "idProveedor_T", "idProveedor").cast("long").alias("IdProveedor_T"),
        fecha(c(planes_raw, "Fecha")).alias("FechaEmision"),
        c(planes_raw, "idNivelServicio_T", "idNivelDeServicio_T", "idNivelServicio", "idNivelDeServicio").cast("long").alias("IdNivelDeServicio_T"),
        c(planes_raw, "valorCopago", "ValorCopago").cast("double").alias("ValorCopago"),
        c(planes_raw, "valorCoseguro", "ValorCoseguro").cast("double").alias("ValorCoseguro"),
        c(planes_raw, "cantidadLimite", "CantidadLimite").cast("double").alias("CantidadLimiteOriginal"),
    )
    .filter(f.col("IdPlan").isNotNull() & f.col("IdTipoBeneficio_T").isNotNull() & f.col("FechaEmision").isNotNull())
    .dropDuplicates()
)

## 4. Dimensiones y minidimensión de condiciones

In [6]:
# Dimensiones básicas
dim_areas = (
    areas.select("IdAreaDeServicio_T", "NombreAreaDeServicio", "Fecha")
    .dropDuplicates(["IdAreaDeServicio_T"])
    .withColumn("IdAreaDeServicio_DWH", f.row_number().over(Window.orderBy("IdAreaDeServicio_T")))
    .select("IdAreaDeServicio_DWH", "IdAreaDeServicio_T", "NombreAreaDeServicio", "Fecha")
)
dim_areas = desconocido(dim_areas, (0, 0, "Desconocido", date(1900, 1, 1)))

dim_geografia = (
    areas.select("IdGeografia_T", "Estado", "Condado", "Area", "Densidad", "PoblacionAct")
    .filter(f.col("IdGeografia_T").isNotNull())
    .dropDuplicates(["IdGeografia_T"])
    .withColumn("IdGeografia_DWH", f.row_number().over(Window.orderBy("IdGeografia_T")))
    .select("IdGeografia_DWH", "IdGeografia_T", "Estado", "Condado", "Area", "Densidad", "PoblacionAct")
)
dim_geografia = desconocido(dim_geografia, (0, 0, "Desconocido", "Desconocido", 0.0, 0.0, 0))

dim_niveles = (
    niveles_fuente.withColumn("IdNivelDeServicio_DWH", f.row_number().over(Window.orderBy("IdNivelDeServicio_T")))
    .select("IdNivelDeServicio_DWH", "IdNivelDeServicio_T", "Descripcion")
)
dim_niveles = desconocido(dim_niveles, (0, 0, "Desconocido"))

dim_pagos = (
    dim_pago_fuente.withColumn("IdCondicionesDePago_DWH", f.row_number().over(Window.orderBy("IdCondicionesDePago_T")))
    .select("IdCondicionesDePago_DWH", "IdCondicionesDePago_T", "Descripcion", "Tipo")
)
dim_pagos = desconocido(dim_pagos, (0, 0, "Desconocido", "Desconocido"))

dim_tipos = (
    beneficios.select("IdTipoBeneficio_T", "Nombre")
    .dropDuplicates(["IdTipoBeneficio_T"])
    .withColumn("IdTipoBeneficio_DWH", f.row_number().over(Window.orderBy("IdTipoBeneficio_T")))
    .select("IdTipoBeneficio_DWH", "IdTipoBeneficio_T", "Nombre")
)
dim_tipos = desconocido(dim_tipos, (0, 0, "Desconocido"))

dim_proveedor = (
    planes.select("IdProveedor_T").filter(f.col("IdProveedor_T").isNotNull()).dropDuplicates()
    .withColumn("IdProveedor_DWH", f.row_number().over(Window.orderBy("IdProveedor_T")))
    .select("IdProveedor_DWH", "IdProveedor_T")
)
dim_proveedor = desconocido(dim_proveedor, (0, 0))

dim_asociacion = (
    dim_areas.alias("a")
    .join(areas.select("IdAreaDeServicio_T", "IdGeografia_T").dropDuplicates().alias("x"), "IdAreaDeServicio_T")
    .join(dim_geografia.alias("g"), f.col("x.IdGeografia_T") == f.col("g.IdGeografia_T"), "left")
    .select(f.col("a.IdAreaDeServicio_DWH"), f.coalesce(f.col("g.IdGeografia_DWH"), f.lit(0)).alias("IdGeografia_DWH"))
    .dropDuplicates()
)

In [7]:
# Historia tipo 4 para las condiciones de los beneficios.
cond_cols = [
    "EstaCubiertoPorSeguro", "EsEHB", "TieneLimiteCuantitativo",
    "ExcluidoDelDesembolsoMaximoDentroDeLaRed",
    "ExcluidoDelDesembolsoMaximoFueraDeLaRed", "UnidadDelLimite",
]
versiones = beneficios.dropDuplicates(["IdTipoBeneficio_T", "FechaInicio"] + cond_cols)
ventana_historia = Window.partitionBy("IdTipoBeneficio_T").orderBy("FechaInicio")
versiones = (
    versiones.withColumn("SiguienteInicio", f.lead("FechaInicio").over(ventana_historia))
    .withColumn("FechaFin", f.when(f.col("SiguienteInicio").isNotNull(), f.date_sub("SiguienteInicio", 1)))
    .drop("SiguienteInicio")
    .withColumn(
        "IdCondicionesTipoBeneficio_T",
        f.concat_ws("|", f.col("IdTipoBeneficio_T"), f.date_format("FechaInicio", "yyyyMMdd"), *[f.coalesce(f.col(x), f.lit("SIN_DATO")) for x in cond_cols]),
    )
)

dim_mini = (
    versiones.withColumn("IdCondicionesTipoBeneficio_DWH", f.row_number().over(Window.orderBy("IdCondicionesTipoBeneficio_T")))
    .select("IdCondicionesTipoBeneficio_DWH", "IdCondicionesTipoBeneficio_T", *cond_cols)
)
dim_mini = desconocido(dim_mini, (0, "0", "Desconocido", "Desconocido", "Desconocido", "Desconocido", "Desconocido", "Desconocido"))

fechas = (
    planes.select(f.col("FechaEmision").alias("Fecha"))
    .unionByName(versiones.select(f.col("FechaInicio").alias("Fecha")))
    .unionByName(versiones.select(f.col("FechaFin").alias("Fecha")))
    .filter(f.col("Fecha").isNotNull()).dropDuplicates()
)
dim_fechas = (
    fechas.withColumn("IdFecha_DWH", f.row_number().over(Window.orderBy("Fecha")))
    .withColumn("Anio", f.year("Fecha")).withColumn("Mes", f.month("Fecha")).withColumn("Dia", f.dayofmonth("Fecha"))
    .select("IdFecha_DWH", "Fecha", "Anio", "Mes", "Dia")
)
dim_fechas = desconocido(dim_fechas, (0, date(1900, 1, 1), 1900, 1, 1))

## 5. Carga de dimensiones

In [8]:
for df, tabla in [
    (dim_mini, DEST["mini"]), (dim_areas, DEST["areas"]), (dim_geografia, DEST["geografia"]),
    (dim_asociacion, DEST["asociacion"]), (dim_niveles, DEST["niveles"]),
    (dim_pagos, DEST["pagos"]), (dim_fechas, DEST["fechas"]),
    (dim_tipos, DEST["tipos"]), (dim_proveedor, DEST["proveedores"]),
]:
    cargar_si_vacia(df, tabla, batchsize=5_000, particiones=1)

DimMiniCondicionesTipoBeneficio: se omite porque ya tiene datos.


DimAreasDeServicio: se omite porque ya tiene datos.


26/07/15 17:48:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 1

DimGeografia: carga completada.


26/07/15 17:48:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 1

...

DimFecha: carga completada.


26/07/15 17:48:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 1

DimTipoBeneficio: carga completada.


26/07/15 17:48:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:48:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


DimProveedor: carga completada.


## 6. Hechos

In [11]:
hecho_historia = (
    versiones.alias("v")
    .join(dim_tipos.alias("t"), f.col("v.IdTipoBeneficio_T") == f.col("t.IdTipoBeneficio_T"), "left")
    .join(dim_mini.alias("m"), f.col("v.IdCondicionesTipoBeneficio_T") == f.col("m.IdCondicionesTipoBeneficio_T"), "left")
    .join(dim_fechas.alias("fi"), f.col("v.FechaInicio") == f.col("fi.Fecha"), "left")
    .join(dim_fechas.alias("ff"), f.col("v.FechaFin") == f.col("ff.Fecha"), "left")
    .select(
        f.coalesce(f.col("t.IdTipoBeneficio_DWH"), f.lit(0)).alias("IdTipoBeneficio_DWH"),
        f.coalesce(f.col("m.IdCondicionesTipoBeneficio_DWH"), f.lit(0)).alias("IdCondicionesTipoBeneficio_DWH"),
        f.coalesce(f.col("fi.IdFecha_DWH"), f.lit(0)).alias("IdFechaInicio"),
        f.coalesce(f.col("ff.IdFecha_DWH"), f.lit(0)).alias("IdFechaFin"),
        f.lit(1).alias("Cambio"),
    ).dropDuplicates()
)

# Busca la condición vigente para la fecha de emisión del plan.
versiones_join = versiones.select(
    f.col("IdTipoBeneficio_T").alias("IdTipoBeneficioVersion_T"),
    f.col("FechaInicio").alias("FechaInicioVersion"),
    f.col("FechaFin").alias("FechaFinVersion"),
    f.col("IdCondicionesTipoBeneficio_T"),
    f.col("TieneLimiteCuantitativo").alias("TieneLimiteCuantitativoVersion"),
)

planes_id = planes.withColumn(
    "IdRegistroPlan",
    f.sha2(
        f.concat_ws(
            "|",
            *[
                f.coalesce(f.col(x).cast("string"), f.lit("SIN_DATO"))
                for x in planes.columns
            ],
        ),
        256,
    ),
)

plan_condicion = (
    planes_id.alias("p")
    .join(
        versiones_join.alias("v"),
        (f.col("p.IdTipoBeneficio_T") == f.col("v.IdTipoBeneficioVersion_T"))
        & (f.col("p.FechaEmision") >= f.col("v.FechaInicioVersion"))
        & (
            f.col("v.FechaFinVersion").isNull()
            | (f.col("p.FechaEmision") <= f.col("v.FechaFinVersion"))
        ),
        "left",
    )
    # Conserva solo las columnas del plan y agrega las de historia con nombres únicos.
    .select(
        "p.*",
        f.col("v.IdCondicionesTipoBeneficio_T"),
        f.col("v.TieneLimiteCuantitativoVersion"),
        f.col("v.FechaInicioVersion"),
    )
)

ventana_plan = Window.partitionBy("IdRegistroPlan").orderBy(
    f.col("FechaInicioVersion").desc_nulls_last()
)

plan_condicion = (
    plan_condicion
    .withColumn("rn", f.row_number().over(ventana_plan))
    .filter(f.col("rn") == 1)
    .drop("rn")
    .withColumn(
        "CantidadLimite",
        f.when(
            (f.col("TieneLimiteCuantitativoVersion") == "Yes")
            & (
                f.col("CantidadLimiteOriginal").isNull()
                | (f.col("CantidadLimiteOriginal") == 0)
            ),
            f.lit(333.0),
        ).otherwise(f.col("CantidadLimiteOriginal")),
    )
    .withColumn(
        "ValorCopago",
        f.when(
            (f.year("FechaEmision") == 2018) & (f.col("ValorCopago") > 3500),
            3500.0,
        ).otherwise(f.col("ValorCopago")),
    )
    .withColumn(
        "ValorCoseguro",
        f.when(
            (f.year("FechaEmision") == 2018) & (f.col("ValorCoseguro") > 100),
            100.0,
        ).otherwise(f.col("ValorCoseguro")),
    )
)

In [12]:
hecho_planes = (
    plan_condicion.alias("p")
    .join(dim_proveedor.alias("pr"), f.col("p.IdProveedor_T") == f.col("pr.IdProveedor_T"), "left")
    .join(dim_fechas.alias("f"), f.col("p.FechaEmision") == f.col("f.Fecha"), "left")
    .join(dim_areas.alias("a"), f.col("p.IdAreaDeServicio_T") == f.col("a.IdAreaDeServicio_T"), "left")
    .join(dim_niveles.alias("n"), f.col("p.IdNivelDeServicio_T") == f.col("n.IdNivelDeServicio_T"), "left")
    .join(dim_tipos.alias("t"), f.col("p.IdTipoBeneficio_T") == f.col("t.IdTipoBeneficio_T"), "left")
    .join(dim_mini.alias("m"), f.col("p.IdCondicionesTipoBeneficio_T") == f.col("m.IdCondicionesTipoBeneficio_T"), "left")
    .join(dim_pagos.alias("coseguro"), f.col("p.IdCondicionesDePagoCoseguro_T") == f.col("coseguro.IdCondicionesDePago_T"), "left")
    .join(dim_pagos.alias("copago"), f.col("p.IdCondicionesDePagoCopago_T") == f.col("copago.IdCondicionesDePago_T"), "left")
    .select(
        f.coalesce(f.col("pr.IdProveedor_DWH"), f.lit(0)).alias("IdProveedor_DWH"),
        f.coalesce(f.col("f.IdFecha_DWH"), f.lit(0)).alias("IdFechaEmision"),
        f.col("p.IdPlan").alias("IdPlan"),
        f.coalesce(f.col("a.IdAreaDeServicio_DWH"), f.lit(0)).alias("IdAreaDeServicio_DWH"),
        f.coalesce(f.col("n.IdNivelDeServicio_DWH"), f.lit(0)).alias("IdNivelDeServicio_DWH"),
        f.coalesce(f.col("t.IdTipoBeneficio_DWH"), f.lit(0)).alias("IdTipoBeneficio_DWH"),
        f.coalesce(f.col("m.IdCondicionesTipoBeneficio_DWH"), f.lit(0)).alias("IdCondicionesTipoBeneficio_DWH"),
        f.coalesce(f.col("coseguro.IdCondicionesDePago_DWH"), f.lit(0)).alias("IdCondicionesDePagoCoseguro_DWH"),
        f.coalesce(f.col("copago.IdCondicionesDePago_DWH"), f.lit(0)).alias("IdCondicionesDePagoCopago_DWH"),
        f.col("p.ValorCoseguro"), f.col("p.ValorCopago"), f.col("p.CantidadLimite"),
    ).dropDuplicates()
)

cargar_si_vacia(hecho_historia, DEST["historia"])
cargar_si_vacia(hecho_planes, DEST["hecho"], batchsize=10_000, particiones=4)

26/07/15 17:55:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 1

HechoHistCondicionesTiposBeneficio: carga completada.


26/07/15 17:55:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 1

HechoPlanesTipoBeneficio: carga completada.


## 7. Validaciones y descripción

In [13]:
assert areas.filter(f.col("Area") < 0).count() == 0
assert beneficios.filter(f.col("TieneLimiteCuantitativo").isNotNull() & ~f.col("TieneLimiteCuantitativo").isin("Yes", "No")).count() == 0
assert dim_pago_fuente.filter(~f.col("Tipo").isin("Copago", "Coseguro")).count() == 0
assert hecho_historia.filter(f.col("Cambio") != 1).count() == 0

for nombre, df in {
    "DimMiniCondicionesTipoBeneficio": dim_mini,
    "DimAreasDeServicio": dim_areas,
    "DimGeografia": dim_geografia,
    "DimAsociacionAreaServicioGeografia": dim_asociacion,
    "DimNivelesDeServicio": dim_niveles,
    "DimCondicionDePago": dim_pagos,
    "DimFecha": dim_fechas,
    "DimTipoBeneficio": dim_tipos,
    "DimProveedor": dim_proveedor,
    "HechoHistCondicionesTiposBeneficio": hecho_historia,
    "HechoPlanesTipoBeneficio": hecho_planes,
}.items():
    print(nombre, df.count())

DimMiniCondicionesTipoBeneficio 346


DimAreasDeServicio 11621


26/07/15 17:55:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


DimGeografia 2647


26/07/15 17:55:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 1

...

DimProveedor 1


26/07/15 17:55:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 1

HechoHistCondicionesTiposBeneficio 345


26/07/15 17:55:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 17:55:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/15 1

HechoPlanesTipoBeneficio 0


El ETL elimina duplicados, corrige áreas negativas, corrige el patrón de población terminado en 0001, transforma fechas a formato estándar y normaliza condiciones de beneficio y pago. En el hecho de planes se aplican los máximos definidos para 2018 y el valor 333 cuando un plan con límite cuantitativo tiene dicho límite vacío o igual a cero.

La minidimensión conserva una fila por versión de condiciones y la tabla factless registra el intervalo de vigencia de cada una con Cambio igual a 1. El hecho de planes relaciona cada emisión con la versión de condiciones vigente en su fecha.